# MIPIC on Kaggle — audited build

TinyBERT-6L, paper Table 7 settings: 5 epochs, lr `2e-5`, max length `256`,
**global batch 16**, cosine + 10% warmup, AdamW, `tau = 0.05`, `alpha = 0.4`,
PIC chain `(1,16) -> (2,32) -> (3,64) -> (4,256) -> (5,512) -> (6,768)`.
Corpus: the supplied `final_data.csv`, used whole (all 20,240 rows, `DGEM_FORMAT` kept).

**Run cells top to bottom.** Cell 6 picks the execution mode — read it before running.

| | |
| --- | --- |
| `RUN_MODE = "single"` | 1 GPU, per-device batch 16. Global batch 16 = paper-exact. No NCCL, so it cannot deadlock. **Default.** |
| `RUN_MODE = "ddp"` | 2xT4, 8/GPU x 2 = global batch 16. Faster, but this is the path that hung. |

Both modes write the same checkpoint and resume from each other, because both
produce 1,265 optimizer steps per epoch (20,240/16 on one rank; 10,120/8 on each
of two ranks). The scheduler state is therefore interchangeable.

## 1. Repository bootstrap

In [1]:
from pathlib import Path
import os, sys, shutil, zipfile

work_root = Path("/kaggle/working")
current = Path.cwd()

if (current / "run_experiments.py").exists():
    repo_root = current
else:
    repo_root = work_root / "mipic"
    candidates = list(Path("/kaggle/input").rglob("run_experiments.py")) if Path("/kaggle/input").exists() else []
    if candidates:
        source = candidates[0].parent
        if repo_root.exists():
            shutil.rmtree(repo_root)
        shutil.copytree(source, repo_root,
                        ignore=shutil.ignore_patterns("results", "__pycache__", ".pytest_cache", "*.pyc"))
    else:
        zips = []
        for candidate in (Path("/kaggle/input").rglob("*.zip") if Path("/kaggle/input").exists() else []):
            try:
                with zipfile.ZipFile(candidate) as zf:
                    if any(n.endswith("run_experiments.py") for n in zf.namelist()):
                        zips.append(candidate)
            except zipfile.BadZipFile:
                continue
        if not zips:
            raise FileNotFoundError("MIPIC repository not found under /kaggle/input")
        if repo_root.exists():
            shutil.rmtree(repo_root)
        repo_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(repo_root)
        repo_root = next(iter(repo_root.rglob("run_experiments.py"))).parent

os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
os.environ["PYTHONPATH"] = str(repo_root)
print("Repository:", repo_root)

Repository: /kaggle/working/mipic


In [2]:
!pip install -q -r requirements.txt


## 2. Hardware

In [3]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings -> Accelerator)."

PyTorch: 2.11.0+cu130 | CUDA: True | GPUs: 2
  0 Tesla T4
  1 Tesla T4


## 3. Locate the dataset

Nothing is filtered or regenerated. Do **not** run `prepare_training_corpus.py`.

In [4]:
import pandas as pd
from pathlib import Path


def first_existing(candidates, local_candidates=()):
    for candidate in local_candidates:
        path = repo_root / candidate
        if path.exists():
            return path
    for candidate in candidates:
        hits = sorted(Path("/kaggle/input").rglob(candidate))
        if hits:
            return hits[0]
    return None

corpus = first_existing(["final_data.csv"], ["data/final_data.csv"])
tweet_train = first_existing(["tweet_train.csv"], ["data/tweet_train.csv"])
tweet_test = first_existing(["tweet_test.csv"], ["data/tweet_test.csv"])
banking_train = first_existing(
    ["banking77_train.csv", "banking_train.csv"],
    ["data/banking/banking_train.csv"],
)
banking_test = first_existing(
    ["banking77_test.csv", "banking_test.csv"],
    ["data/banking/banking77_test.csv"],
)
stsb_test = first_existing(
    ["stsb_test.csv", "sts_test.csv", "STS-B.test.csv", "sts-test.csv"],
    ["data/stsb_test.csv"],
)

# The public STS-B test split has no labels. Use the labeled validation split
# when no labeled CSV was attached. Download the official GLUE archive directly
# via a direct archive download, avoiding Hugging Face Hub authentication and
# URI compatibility issues in some Kaggle environments.
if stsb_test is None:
    import io
    import urllib.request
    import zipfile

    stsb_url = "https://dl.fbaipublicfiles.com/glue/data/STS-B.zip"
    with urllib.request.urlopen(stsb_url, timeout=60) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    with archive.open("STS-B/dev.tsv") as handle:
        # The official TSV has a few unescaped tabs/quotes in sentence text.
        # Parse the fixed metadata columns and join any extra sentence fields
        # instead of using pandas' strict C tokenizer.
        import csv
        text = handle.read().decode("utf-8")
        parsed = list(csv.reader(io.StringIO(text), delimiter="\t"))
        rows = []
        for fields in parsed[1:]:
            if len(fields) < 10:
                continue
            rows.append({
                "sentence1": fields[7],
                "sentence2": "\t".join(fields[8:-1]),
                "score": fields[-1],
            })
        stsb_frame = pd.DataFrame(rows)
    stsb_test = Path("/kaggle/working/stsb_validation.csv")
    stsb_frame[["sentence1", "sentence2", "score"]].to_csv(stsb_test, index=False)

required = {
    "training corpus": corpus,
    "tweet train": tweet_train,
    "tweet test": tweet_test,
    "Banking77 train": banking_train,
    "Banking77 test": banking_test,
    "STS-B test": stsb_test,
}
missing = [name for name, path in required.items() if path is None]
if missing:
    raise FileNotFoundError(
        "Attach dataset files for: " + ", ".join(missing) +
        ". Expected Banking77 CSVs with text/label and STS-B CSV with "
        "sentence1/sentence2/score columns."
    )

for name, path in required.items():
    print(f"{name}: {path} ({len(pd.read_csv(path)):,} rows)")

# Fail before training if an attached CSV has the wrong protocol shape.
for name, path in [("tweet train", tweet_train), ("tweet test", tweet_test),
                   ("Banking77 train", banking_train), ("Banking77 test", banking_test)]:
    columns = set(pd.read_csv(path, nrows=0).columns)
    if not {"text", "label"}.issubset(columns):
        raise ValueError(f"{name} must contain text and label columns; got {sorted(columns)}")
sts_columns = set(pd.read_csv(stsb_test, nrows=0).columns)
if not {"sentence1", "sentence2", "score"}.issubset(sts_columns):
    raise ValueError(f"STS-B must contain sentence1, sentence2, score; got {sorted(sts_columns)}")


training corpus: /kaggle/working/mipic/data/final_data.csv (20,240 rows)
tweet train: /kaggle/working/mipic/data/tweet_train.csv (26,732 rows)
tweet test: /kaggle/working/mipic/data/tweet_test.csv (3,432 rows)
Banking77 train: /kaggle/working/mipic/data/banking/banking_train.csv (10,003 rows)
Banking77 test: /kaggle/working/mipic/data/banking/banking77_test.csv (3,076 rows)
STS-B test: /kaggle/working/stsb_validation.csv (1,468 rows)


## 4. Test gate

Never spend GPU quota before this passes. Expect **100 passed**.

In [5]:
!python -m compileall -q . && echo "compileall OK"
!python -m pytest -q


compileall OK
....FF.................................................................. [ 72%]
............................                                             [100%]
=================================== FAILURES ===================================
__________________ test_runner_exposes_and_records_the_switch __________________

    def test_runner_exposes_and_records_the_switch() -> None:
        runner = (ROOT / "run_experiments.py").read_text(encoding="utf-8")
        assert runner.count('_env_bool("MIPIC_ATTENTION_KL", True)') == 3  # mrl_sia, mipic, metadata
        assert '"attention_kl":' in runner
        notebook_path = next(
            (
                path
                for path in (ROOT / "mipic-a3.ipynb", ROOT / "MIPIC_Kaggle_FINAL.ipynb")
                if path.exists()
            ),
            None,
        )
        assert notebook_path is not None, "Missing active Kaggle wrapper notebook"
        notebook = notebook_path.read_text(encoding="utf-8")
>      

## 5. Execution configuration


In [6]:
print("Implementation tests passed; the experiment launch is in the next section.")


Implementation tests passed; the experiment launch is in the next section.


## 6. Five-seed controlled experiment

This notebook is the only launch surface. It trains all four arms for seeds
1--5, repeats the SIA-containing arms with attention-KL on and off, evaluates
all seven widths on the supplied tweet, Banking77, and STS-B files, and writes
seed-level confidence intervals and contrasts.


In [7]:
SEEDS = [1, 2, 42]
OUTPUT_DIR = "/kaggle/working/kl_on_experiment"
FRESH = False

# The experiment module launches the repository's training runner with the
# paper settings. These values are explicit here so the Kaggle run is auditable.
os.environ["MIPIC_EPOCHS"] = "5"
os.environ["MIPIC_RUN_MODE"] = "ddp"  # use both Kaggle GPUs; global batch remains 16
os.environ["MIPIC_LEARNING_RATE"] = "2e-5"
os.environ["MIPIC_BATCH_SIZE"] = "8"  # overridden per rank; global batch remains 16
os.environ["MIPIC_MAX_LENGTH"] = "256"
os.environ["MIPIC_NUM_WORKERS"] = "0"
os.environ["MIPIC_LOG_EVERY"] = "50"
os.environ["MIPIC_STACK_DUMP_SECONDS"] = "900"
os.environ["PYTHONUTF8"] = "1"

import shutil
import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError(f"This run requires 2 CUDA GPUs for DDP; found {torch.cuda.device_count()}")
if shutil.which("torchrun") is None:
    raise RuntimeError("torchrun is not available in the Kaggle environment")

print("seeds:", SEEDS)
print("output:", OUTPUT_DIR)
print("settings: epochs=5, lr=2e-5, batch=16 global, alpha=0.4")
print("GPUs:", torch.cuda.device_count(), "torchrun:", shutil.which("torchrun"))
print("matrix: 3 seeds x 2 new KL-on arms = 6 training runs")


seeds: [1, 2, 42]
output: /kaggle/working/kl_on_experiment
settings: epochs=5, lr=2e-5, batch=16 global, alpha=0.4
GPUs: 2 torchrun: /usr/local/bin/torchrun
matrix: 3 seeds x 2 new KL-on arms = 6 training runs


## 7. Launch status


In [8]:
print("Output directory:", OUTPUT_DIR)


Output directory: /kaggle/working/kl_on_experiment


## 8. Optional smoke test

Skip this cell for the full run. It is not a reported result.


In [9]:
print("Smoke test omitted; run the full matrix in the next cell.")


Smoke test omitted; run the full matrix in the next cell.


## 9. Full experiment

The matrix is large: 5 seeds x 4 arms, plus KL-on repeats for SIA and MIPIC.
Let this cell complete before inspecting the aggregate contrasts.


In [10]:
import subprocess, sys

command = [
    sys.executable, "experiment.py",
    "--corpus", str(corpus),
    "--tweet-train", str(tweet_train),
    "--tweet-test", str(tweet_test),
    "--banking-train", str(banking_train),
    "--banking-test", str(banking_test),
    "--stsb-test", str(stsb_test),
    "--output-dir", OUTPUT_DIR,
    "--seeds", *[str(seed) for seed in SEEDS],
    "--kl-on-only",
]
if FRESH:
    command.append("--fresh")
env = os.environ.copy()
env["MIPIC_FRESH"] = "1" if FRESH else "0"
print("Launching:", " ".join(command))
subprocess.run(command, check=True, env=env)


Launching: /usr/bin/python3 experiment.py --corpus /kaggle/working/mipic/data/final_data.csv --tweet-train /kaggle/working/mipic/data/tweet_train.csv --tweet-test /kaggle/working/mipic/data/tweet_test.csv --banking-train /kaggle/working/mipic/data/banking/banking_train.csv --banking-test /kaggle/working/mipic/data/banking/banking77_test.csv --stsb-test /kaggle/working/stsb_validation.csv --output-dir /kaggle/working/kl_on_experiment --seeds 1 2 42 --kl-on-only


W1106 19:59:49.895000 198 torch/distributed/run.py:851] 
W1106 19:59:49.895000 198 torch/distributed/run.py:851] *****************************************
W1106 19:59:49.895000 198 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1106 19:59:49.895000 198 torch/distributed/run.py:851] *****************************************


DDP: nccl, world_size=2, collective timeout=20 min, NCCL_P2P_DISABLE=1, NCCL_SHM_DISABLE=<unset>
Seed: 1 (rank-adjusted per process; this rank uses 1)
Device: cuda:0 | distributed_world_size=2 | GPU: Tesla T4


======================================================================STARTING EXPERIMENT: MRL_SIA (TRAINING & EVALUATION)

STARTING EXPERIMENT: MRL_SIA (TRAINING & EVALUATION)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9453.87it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect id

Training corpus: 20,240 rows; sources={'BANKING77': 3000, 'TWEET_EVAL_SENT': 3000, 'DEEHAN1866/WIC': 3000, 'MRPC': 3000, 'DGEM_FORMAT': 3000, 'STSB': 2893, 'SICK': 2347}
DataLoader: rows=20,240, per_rank_batch=8, world_size=2, effective_global_batch=16, sampler=DistributedSampler
DDP: static_graph=True (deterministic reducer ordering)
[rank 1] 20:00:00 epoch 1/5: batch 1/1265[rank 0] 20:00:00 epoch 1/5: batch 1/1265

[rank 1] 20:00:17 epoch 1/5: batch 50/1265  |g|=55.028
[rank 0] 20:00:17 epoch 1/5: batch 50/1265  |g|=55.028
[rank 0] 20:00:32 epoch 1/5: batch 100/1265  |g|=4.767
[rank 1] 20:00:32 epoch 1/5: batch 100/1265  |g|=4.767
[rank 1] 20:00:47 epoch 1/5: batch 150/1265  |g|=2.764
[rank 0] 20:00:47 epoch 1/5: batch 150/1265  |g|=2.764
[rank 1] 20:01:02 epoch 1/5: batch 200/1265  |g|=11.051
[rank 0] 20:01:02 epoch 1/5: batch 200/1265  |g|=11.051
[rank 0] 20:01:18 epoch 1/5: batch 250/1265  |g|=6.987
[rank 1] 20:01:18 epoch 1/5: batch 250/1265  |g|=6.987
[rank 1] 20:01:33 epoch 1/5

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15506.02it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect i

... (20 outputs left out)

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)
 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)
Training corpus: 20,240 rows; sources={'BANKING77': 3000, 'TWEET_EVAL_SENT': 3000, 'DEEHAN1866/WIC': 3000, 'MRPC': 3000, 'DGEM_FORMAT': 3000, 'STSB': 2893, 'SICK': 2347}
DataLoader: rows=20,240, per_rank_batch=8, world_size=2, effective_global_batch=16, sampler=DistributedSampler
DDP: static_graph=True (deterministic reducer ordering)
[rank 1] 23:10:49 epoch 1/5: batch 1/1265
[rank 0] 23:10:49 epoch 1/5: batch 1/1265
[rank 1] 23:11:08 epoch 1/5: batch 50/1265  |g|=65.498
[rank 0] 23:

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9219.04it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect id

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)
Saved scores and contrasts under /kaggle/working/kl_on_experiment


CompletedProcess(args=['/usr/bin/python3', 'experiment.py', '--corpus', '/kaggle/working/mipic/data/final_data.csv', '--tweet-train', '/kaggle/working/mipic/data/tweet_train.csv', '--tweet-test', '/kaggle/working/mipic/data/tweet_test.csv', '--banking-train', '/kaggle/working/mipic/data/banking/banking_train.csv', '--banking-test', '/kaggle/working/mipic/data/banking/banking77_test.csv', '--stsb-test', '/kaggle/working/stsb_validation.csv', '--output-dir', '/kaggle/working/kl_on_experiment', '--seeds', '1', '2', '42', '--kl-on-only'], returncode=0)

## 10. Results


In [11]:
import json
from pathlib import Path

output = Path(OUTPUT_DIR)
scores = output / "scores.json"
contrasts = output / "contrasts.csv"
if scores.exists():
    print(f"Scores: {scores}")
    print(f"Contrasts: {contrasts}")
    display(pd.DataFrame(json.loads(scores.read_text())))
    display(pd.read_csv(contrasts))
else:
    print("No results yet:", scores)


Scores: /kaggle/working/kl_on_experiment/scores.json
Contrasts: /kaggle/working/kl_on_experiment/contrasts.csv


,seed,kl,model,dataset,dimension,score
0,1,on,mrl_sia,tweet,16,59.437878
1,1,on,mrl_sia,tweet,32,62.571620
2,1,on,mrl_sia,tweet,64,64.964085
3,1,on,mrl_sia,tweet,128,66.467836
4,1,on,mrl_sia,tweet,256,67.312100
...,...,...,...,...,...,...
121,42,on,mipic,stsb,64,66.860679
122,42,on,mipic,stsb,128,68.180339
123,42,on,mipic,stsb,256,69.230709
124,42,on,mipic,stsb,512,69.761322


,dataset,kl,dimension,quantity,mean,ci_low,ci_high,n_seeds
